<a href="https://colab.research.google.com/github/ashok-bisht/Context-Aware_Misinformation_Detection_ML_and_Gen_AI/blob/main/notebooks/1.0-eda-data-cleaning_big_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. EDA, Load and Merge

* Load and read df_true and df_fake data.
* View the description of the true and fake sets.
* Label for the true and fake sets (1, 0).

## Clean the data:

* Concat the two datasets into df.
* Remove unused columns, keeping only the title, text and label.
* Remove missing rows with drop null.
* Remove extra spaces.
* Check if the number of real/fake records is equal.
* Mix the data to ensure training.
* Calculate the length of each title in a data point.
* Remove the publisher info like Reuter, CNN etc from the text from both dataset.


In [2]:
#Initialize folder paths
import os
import re
import pandas as pd

# 1. Define the base directory path on your Google Drive
base_drive_folder = "/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection"

# Define explicit input, temp and output folder paths
input_folder_path = os.path.join (base_drive_folder, "input")
temp_folder_path = os.path.join (base_drive_folder, "temp")
clean_folder_path = os.path.join (base_drive_folder, "clean")

input_file_true = os.path.join(input_folder_path, "True.csv")
input_file_fake = os.path.join(input_folder_path, "Fake.csv")
input_temp_true = os.path.join(temp_folder_path, "True_Pub_Cleaned.csv")
input_temp_fake = os.path.join(temp_folder_path, "Fake_Pub_Cleaned.csv")
input_file_combined = os.path.join(input_folder_path, "WELFake_Dataset.csv")
iput_temp_combined = os.path.join(temp_folder_path, "WELFake_Dataset_Cleaned.csv")


## clean datasets by removing the publisher info

In [4]:

# Define the regex cleaning function
def strip_journalism_fingerprints(text, publishers_to_scrub=None):
    if not isinstance(text, str):
        return ""

    # 1. Remove the last line like 'Featured image via Al Drago-Pool/Getty Images
    # We replace it with a single period to preserve the end of the previous sentence!
    text=re.sub(r"\.\s*featured\s+image.*$", ".", text, flags=re.IGNORECASE | re.MULTILINE)
    # Remove [VIDEO], [video], [ Video ], etc., along with any surrounding spaces
    text= re.sub(r"\s*\[\s*video\s*\]\s*", " ", text, flags=re.IGNORECASE).strip()

    # Clean up spacing and weird characters/newlines at the start
    text = text.strip()
    text = text.replace("\u00a0", " ")  # Fix NBSP

    # Standardize punctuation
    punctuation_map = {
        "’": "'", "‘": "'",  # Curly apostrophes -> Straight apostrophe
        "“": '"', "”": '"',  # Curly quotes -> Straight quotes
        "–": "-", "—": "-",  # En/Em dashes -> Standard hyphen
    }
    for curly, straight in punctuation_map.items():
        text = text.replace(curly, straight)
    text = text.replace("â€™", "'")      # Fix broken UTF-8 apostrophes ("â€™" -> "'")

    # Fix "isn t" -> "isn't", "don t" -> "don't"
    text = re.sub(r"\b(isn|don|didn|doesn|can|wasn|weren|haven|hasn|hadn|won|wouldn|shouldn|couldn|aren)\s+t\b", r"\1't", text, flags=re.IGNORECASE)

    # Fix spaces around real apostrophes like "don ' t" or "don 't"
    text = re.sub(r"\b(don|didn|doesn|isn|can)\s*'\s*t\b", r"\1't", text, flags=re.IGNORECASE)

    # 2. Remove leading bracketed corrections/clarifications at the very start
    # Matches: "(In 2nd paragraph...)"
    leading_bracket_pattern = r"^\([^)]+\)\s*"
    text = re.sub(leading_bracket_pattern, "", text)

    # 3. Strip standard datelines (e.g., "NEW YORK (Reuters) - ")
    dateline_pattern = r"^[A-Z\s,]+(?:\s\([^)]+\))?\s*[\s\-\-–—]\s*"
    text = re.sub(dateline_pattern, "", text)

    # 4. Remove standalone bracketed publisher or author markers (e.g., "(Reuters)" or "By Terray Sylvester")
    # This specifically target patterns like "(Reuters)" or "By John Doe (Reuters)" left over in the text
    byline_pattern = r"(?:By\s+[A-Za-z\s]+)?\s*\([^)]+\)"
    text = re.sub(byline_pattern, "", text)

    # 5. Scrub specific publisher words anywhere else in the text
    if publishers_to_scrub:
        escaped_publishers = [re.escape(pub) for pub in publishers_to_scrub]
        scrub_pattern = r"\b(" + "|".join(escaped_publishers) + r")\b"
        text = re.sub(scrub_pattern, "XYZ", text, flags=re.IGNORECASE)

    # Final cleanup of double spaces left behind by removals
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ============================================================
# EXECUTION PIPELINE
# ============================================================
try:

    #******************************************************
    # Process big news file
    #******************************************************

    # Load the original Kaggle CSV file, ignore first unnamed column which has a series of numbers
    print(f"🔄 Loading big news dataset from: {input_file_combined}")
    df_combined= pd.read_csv(input_file_combined, encoding='utf-8', usecols=['title','text','label'])
    print("✂️ Stripping datelines, publisher signatures, and non relevant text...")

    # Apply the regex function exclusively to the text column
    publishers = ["Reuters", "CNN", "Associated Press", "BBC"]
    df_combined["text"] = df_combined["text"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))
    # On title column
    df_combined["title"] = df_combined["title"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))

    # Save to the new destination file name
    print(f"💾 Saving cleaned file to: {temp_folder_path}")
    df_combined.to_csv(iput_temp_combined, index=False)

    print("✅ Process complete! The data has been cleaned.")

except FileNotFoundError:
    print(
        f"❌ Error: Could not find 'WELFake.csv' at '{input_folder_path}'."
        " Please make sure your Google Drive is mounted using "
        " 'from google.colab import drive; drive.mount(\"/content/drive\")'"
    )
except Exception as e:
    print(f"❌ An error occurred during processing: {str(e)}")


🔄 Loading big news dataset from: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/input/WELFake_Dataset.csv
✂️ Stripping datelines, publisher signatures, and non relevant text...
💾 Saving cleaned file to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/temp
✅ Process complete! The data has been cleaned.


## Change the column name text -> content and merge dataset


In [5]:
import numpy as np # linear algebra
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Rename 'text' column to 'content' for consistency
df_combined.rename(columns={'text': 'content'}, inplace=True)

# Display the first few rows and info of the combined dataframe
print("Combined DataFrame Head:")
display(df_combined.head())
print("\nCombined DataFrame Info:")
df_combined.info()
df_combined.describe()

Combined DataFrame Head:


,title,content,label
0,Following Threats Against Cops And Whites On 9...,No comment is expected from Barack Obama Membe...,1
1,,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA'S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last n...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",dozen politically active pastors came here for...,0
4,2: Russia unvelis an image of its terrifying n...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1



Combined DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72134 entries, 0 to 72133
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    72134 non-null  object
 1   content  72134 non-null  object
 2   label    72134 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.7+ MB


,label
count,72134.000000
mean,0.514404
std,0.499796
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


In [6]:
#Drop NA records
df_combined.dropna(inplace=True)
print ("After Drop NA",df_combined.shape)
#Trim spaces at end
df_combined['title'] = df_combined['title'].astype(str).str.strip()
df_combined['content'] = df_combined['content'].astype(str).str.strip()
print ("After Trim Space", df_combined.shape)
#remove the duplicate records
df_combined.drop_duplicates(inplace=True)
print ("After Duplicate removal", df_combined.shape)


After Drop NA (72134, 3)
After Trim Space (72134, 3)
After Duplicate removal (63670, 3)


In [7]:
#check true vs fake
print ("Check the distribution of True and False")
print(df_combined["label"].value_counts())

Check the distribution of True and False
label
0    34788
1    28882
Name: count, dtype: int64


In [8]:
# Randomly mix the data
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_combined.head()

,title,content,label
0,Watch Joe Biden Tells Us EXACTLY What Trump Is...,With more and more coming out about Republican...,0
1,New Research Shows Trump's Border Wall Will Be...,Build the wall! Build the wall! Supposedly nob...,0
2,Trump seeks to bar personal conduct claims fro...,Donald Trump's attorneys asked a U.S. judge to...,1
3,Turkey's Erdogan: Iraqi Kurds' decision not to...,Iraqi Kurdish leader Massoud Barzani s decisio...,1
4,"After Being Handed Yet Another Court Loss, Tru...",Donald Trump hates losing. Not because it s a ...,0


# Temp - update the Title with XYZ and check if results are better


In [ ]:
#update titel from df_combined to "XYZ"
# df_combined['title'] = "XYZ"
# df_combined.head()

## Save Cleaned Dataset to the google drive Path


In [ ]:
# Save the df_combined to this location
import os

# 1. Define the Google Drive folder and file name
combined_csv_path = os.path.join(clean_folder_path, 'combined_news.csv')

# 2. Ensure the directory exists
os.makedirs(clean_folder_path, exist_ok=True)

print("Saving DataFrame to Google Drive... Please wait.")

# 3. Save to CSV (index=False prevents pandas from adding an extra row numbers column)
df_combined.to_csv(combined_csv_path, index=False)

print(f"🎉 Success! Dataset successfully saved to: {combined_csv_path}")



Saving DataFrame to Google Drive... Please wait.
🎉 Success! Dataset successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/clean/combined_news.csv


In [ ]:
# Install spaCy and download the English model
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 97.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 2. Text Cleaning with spaCy

Now, clean the text data using spaCy. This involves:
-   **Loading the spaCy model**: `en_core_web_sm`.
-   **Tokenization**: Breaking down text into individual words.
-   **Stopword Removal**: Removing common words that don't add much meaning.
-   **Lowercasing**: Converting all text to lowercase.
-   **Removing Punctuation and Special Characters**.
-   **Lemmatization**: Reducing words to their base form.

After cleaning, save the processed DataFrame to a new CSV file named `cleaned_news_data.csv`.

In [ ]:
# run if the colab fails in middle.
#load the DF from the drive
# import pandas as pd
# #pd.set_option('display.max_colwidth', 50)
# df_combined = pd.read_csv(combined_csv_path)
# df_combined.head()

In [ ]:
import os
from datetime import datetime
import pandas as pd
import spacy

# ==========================================
# ⚙️ CONFIGURATION BLOCK
# ==========================================
START_ROW = 0     # Change this to resume (e.g., 15000) if it fails midway
N = 1000          # Configurable 'n' rows per batch, CSV append, and log update
# ==========================================

# 1. Setup Folder and File Paths
spacy_csv_file = os.path.join(clean_folder_path, 'cleaned_news_spacy.csv')
if os.path.exists(spacy_csv_file):
    os.remove(spacy_csv_file)
    print(f"Successfully deleted existing old file {spacy_csv_file}")

# 2. Load spaCy with optimized disabling to maximize performance
nlp = spacy.load('en_core_web_sm', disable=['tok2vec', 'parser', 'ner'])

# 5. Generate Log File with Unique Timestamp Suffix
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"progress_log_{current_time}.txt"
log_filepath = os.path.join(clean_folder_path, log_filename)

# Reusable helper function to process a list of texts through nlp.pipe
def clean_text_list(text_list):
    cleaned = []
    for doc in nlp.pipe(text_list, batch_size=250, n_process=-1):
        tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop and not token.pos_ == "PROPN"]
        cleaned.append(" ".join(tokens))
    return cleaned

total_rows = len(df_combined)
print(f"Starting processing from row {START_ROW} out of {total_rows} total rows (Batch size N = {N})...")



# 3. Process and Save in Configurable Batches
for start in range(START_ROW, total_rows, N):
    end = min(start + N, total_rows)

    # Slice the current chunk from the main DataFrame
    batch_df = df_combined.iloc[start:end].copy()

    # --- PROCESS COLUMN 1: TITLE ---
    title_texts = batch_df['title'].fillna("").astype(str).str.lower().tolist()
    batch_df['cleaned_title'] = clean_text_list(title_texts)

    # --- PROCESS COLUMN 2: CONTENT ---
    content_texts = batch_df['content'].fillna("").astype(str).str.lower().tolist()
    batch_df['cleaned_content'] = clean_text_list(content_texts)

    # 4. Save to CSV in Append Mode
    if start == 0 and not os.path.exists(spacy_csv_file):
        batch_df.to_csv(spacy_csv_file, index=False, mode='w')
    else:
        # Append mode ('a') bypasses writing the column headers again
        batch_df.to_csv(spacy_csv_file, index=False, mode='a', header=False)

    log_content = (
        f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"Status: Success\n"
        f"Last Completed Index: {end - 1}\n"
        f"Processed Rows: {start} to {end - 1}\n"
        f"Total Data Progress: {end}/{total_rows} rows complete.\n"
    )

    with open(log_filepath, "a") as log_file:
        log_file.write(log_content)

    print(f"✅ Appended rows {start} to {end-1} (Title, Content cleaned). Log: {log_filename}")

print(f"\n🎉 All processing completed! Final file saved with multiple clean features at: {spacy_csv_file}")


Successfully deleted existing old file /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/clean/cleaned_news_spacy.csv
Starting processing from row 0 out of 39100 total rows (Batch size N = 1000)...
✅ Appended rows 0 to 999 (Title, Content cleaned). Log: progress_log_20260805_124644.txt
✅ Appended rows 1000 to 1999 (Title, Content cleaned). Log: progress_log_20260805_124644.txt
✅ Appended rows 2000 to 2999 (Title, Content cleaned). Log: progress_log_20260805_124644.txt
✅ Appended rows 3000 to 3999 (Title, Content cleaned). Log: progress_log_20260805_124644.txt
✅ Appended rows 4000 to 4999 (Title, Content cleaned). Log: progress_log_20260805_124644.txt
✅ Appended rows 5000 to 5999 (Title, Content cleaned). Log: progress_log_20260805_124644.txt
✅ Appended rows 6000 to 6999 (Title, Content cleaned). Log: progress_log_20260805_124644.txt
✅ Appended rows 7000 to 7999 (Title, Content cleaned). Log: progress_log_20260805_124644.txt
✅ Appended rows 8000 to 8999 (Titl

### 3. TF-IDF Vectorization

Finally, perform TF-IDF (Term Frequency-Inverse Document Frequency) vectorization on the `cleaned_content` column. TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus.

I will use `TfidfVectorizer` from `sklearn.feature_extraction.text` to convert the text data into a matrix of TF-IDF features. The resulting TF-IDF matrix will be saved as a new CSV file named `tfidf_vectors.csv` for use by other team members.

In [ ]:
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import numpy as np

# 1. Load the cleaned data
df_spacy = pd.read_csv(spacy_csv_file)

# 2. Critical Step: Fill empty/missing cells with blank strings to prevent crashes
df_spacy['cleaned_title'] = df_spacy['cleaned_title'].fillna("")
df_spacy['cleaned_content'] = df_spacy['cleaned_content'].fillna("")

# 3. Initialize separate vectorizers using a ColumnTransformer
# We apply different max_feature caps based on typical text length per field
preprocessor = ColumnTransformer(
    transformers=[
        ('title_tfidf', TfidfVectorizer(max_features=5000), 'cleaned_title'),
        ('content_tfidf', TfidfVectorizer(max_features=25000), 'cleaned_content')
    ]
)

print("Vectorizing features in parallel...")
# 4. Transform your columns into a horizontal combined sparse matrix
tfidf_matrix = preprocessor.fit_transform(df_spacy)

# 5. Extract unique, descriptive names for all newly generated feature columns
feature_names = preprocessor.get_feature_names_out()

# Define output file paths

sparse_matrix_file_path = os.path.join(base_drive_folder, 'tfidf_sparse_matrix.npz')
feature_names_file_path = os.path.join(base_drive_folder, 'tfidf_feature_names.npy')
labels_file_path = os.path.join(base_drive_folder, 'tfidf_labels.csv')

# 6. Save the sparse matrix using scipy.sparse.save_npz
scipy.sparse.save_npz(sparse_matrix_file_path, tfidf_matrix)
print(f"\nSparse TF-IDF matrix successfully saved to: {sparse_matrix_file_path}")

# 7. Save feature names (as a numpy array)
np.save(feature_names_file_path, feature_names)
print(f"TF-IDF feature names successfully saved to: {feature_names_file_path}")

# 8. Save the classification labels separately
df_spacy['label'].to_csv(labels_file_path, index=False)
print(f"Classification labels successfully saved to: {labels_file_path}")


Vectorizing features in parallel...

Sparse TF-IDF matrix successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_sparse_matrix.npz
TF-IDF feature names successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_feature_names.npy
Classification labels successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_labels.csv
